# 07 — Results Dashboard

Every figure your paper and defense slides need, rendered from the artifacts
notebooks 00–06 already wrote. Nothing is retrained here, so this runs on CPU
in a couple of minutes and you can re-run it every time an upstream number
changes.

Each section is independent and skips itself with a message if its inputs are
missing, so a partial pipeline still produces everything it can.

| Section | Figure | Use it for |
|---|---|---|
| 1 | Dataset overview | Section 3, Dataset |
| 2 | Sample frames | Figure 1, qualitative |
| 3 | Model comparison with CI | Section 6, main result |
| 4 | ROC and PR curves | Section 6 / supplementary |
| 5 | Per-class F1 heatmap | Section 6, where it works |
| 6 | Error decomposition | Section 9, Discussion |
| 7 | Score distributions | Calibration, supplementary |
| 8 | Retrieval quality | Section 6, RAG |
| 9 | Generation metrics | Section 6, RAG |
| 10 | Embedding space | Section 5 / supplementary |
| 11 | MIL attention | Section 8, Explainability |
| 12 | Qualitative examples | Figure, case studies |

Everything lands in `artifacts/figures/paper/` at 300 dpi, plus a single
`all_figures.zip` you can download in one click.

In [1]:
import os
import sys
import json
import re
import zipfile
import warnings
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures" / "paper"
FIG.mkdir(parents=True, exist_ok=True)

# A restrained palette that survives greyscale printing and colour-blind readers.
C = {"primary": "#3B6EA5", "accent": "#C05746", "green": "#4F8A6B",
     "amber": "#D6A34A", "grey": "#8A8F98", "dark": "#2B2F36"}
TIER_C = {"trainable": C["primary"], "marginal": C["amber"], "rare": C["grey"]}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "semibold",
    "axes.labelsize": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.facecolor": "white",
})

SAVED = []


def save(fig, name, caption=""):
    path = FIG / f"{name}.png"
    fig.savefig(path)
    fig.savefig(FIG / f"{name}.pdf")          # vector copy for LaTeX
    plt.show()
    SAVED.append({"figure": name, "file": path.name, "caption": caption})
    print(f"  saved {name}")


def find_file(fname):
    for base in (ART, ART / "embeddings"):
        if (base / fname).exists():
            return base / fname
    hits = sorted(Path("/kaggle/input").rglob(fname))
    return hits[0] if hits else None


def load_csv(fname):
    p = find_file(fname)
    return pd.read_csv(p) if p is not None else None


def load_json(fname):
    p = find_file(fname)
    return json.loads(p.read_text()) if p is not None else None


def skip(section, missing):
    print(f"[skip] {section} — missing {missing}. Run the notebook that produces it.")

In [2]:
df = load_csv("master_split.csv")
assert df is not None, "master_split.csv not found — run notebooks 00-02 first."
frames = load_csv("frames.csv")
tiers = load_json("label_tiers.json") or {}
Ydf = load_csv("labels_multihot.csv")

TARGETS = [t for t in (tiers.get("trainable", []) + tiers.get("marginal", []))
           if Ydf is not None and t in Ydf.columns]
Y = Ydf[TARGETS].values.astype(np.float32) if TARGETS else None
folds = df["fold"].values if "fold" in df.columns else None
study_ids = df["study_id"].values

MODELS, THRESH = {}, {}
for p in sorted(set(list(ART.glob("oof_*.npy")) +
                    [q for q in Path("/kaggle/input").rglob("oof_*.npy")])):
    name = p.stem.replace("oof_", "")
    arr = np.load(p)
    if Y is not None and arr.shape == Y.shape:
        MODELS[name] = arr
        tp = p.parent / f"thresholds_{name}.npy"
        THRESH[name] = np.load(tp) if tp.exists() else np.full(len(TARGETS), 0.5)

print(f"{len(df)} studies | {len(TARGETS)} targets | "
      f"{len(MODELS)} models: {', '.join(MODELS) or 'none'}")

PRETTY = {"prior": "Prior", "majority": "Majority", "tfidf_logreg": "TF-IDF + LR",
          "tfidf_svm": "TF-IDF + SVM", "clinicalbert": "ClinicalBERT",
          "img_resnet50": "ResNet50 (MIL)", "img_densenet121": "DenseNet121 (MIL)",
          "img_effnetv2s": "EfficientNetV2-S (MIL)", "fusion_late": "Late fusion",
          "zeroshot": "BiomedCLIP zero-shot", "probe_image": "BiomedCLIP probe (image)",
          "probe_text_findings": "BiomedCLIP probe (text)",
          "probe_concat": "BiomedCLIP probe (both)", "multitask": "Multi-task VLM"}
nice = lambda m: PRETTY.get(m, m)


def binarise(name):
    return (MODELS[name] >= THRESH[name]).astype(int)


def best_model():
    """Highest macro-F1 among models that actually learned something."""
    from sklearn.metrics import f1_score
    real = [m for m in MODELS if m not in ("prior", "majority")]
    if not real:
        return None
    return max(real, key=lambda m: f1_score(Y, binarise(m), average="macro",
                                            zero_division=0))


BEST = best_model()
print("Best model:", nice(BEST) if BEST else "n/a")

259 studies | 13 targets | 14 models: clinicalbert, fusion_late, img_densenet121, img_effnetv2s, img_resnet50, majority, prior, tfidf_logreg, tfidf_svm, multitask, probe_concat, probe_image, probe_text_findings, zeroshot
Best model: Multi-task VLM


## 1. Dataset overview

The four panels a reviewer looks for before reading any result: what the
classes are, how imbalanced they are, whether the folds are balanced, and how
long the reports run.

In [3]:
freq = load_csv("label_frequency.csv")
if freq is None and Ydf is not None:
    num = Ydf.select_dtypes("number")
    freq = pd.DataFrame({"label": num.columns, "count": num.sum(0).values})

if freq is not None:
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.75, wspace=0.28,
                  height_ratios=[1.0, 1.15])

    # (a) label frequency, coloured by modelling tier
    ax = fig.add_subplot(gs[0, :])
    fr = freq.sort_values("count", ascending=False)
    tier_of = {}
    for t, names in tiers.items():
        if isinstance(names, list):
            for n in names:
                tier_of[n] = t
    cols = [TIER_C.get(tier_of.get(l, "rare"), C["grey"]) for l in fr["label"]]
    ax.bar(range(len(fr)), fr["count"], color=cols)
    ax.set_xticks(range(len(fr)))
    ax.set_xticklabels(fr["label"], rotation=55, ha="right", fontsize=7.5)
    ax.set_ylabel("studies")
    ax.set_title("(a) Disease label distribution")
    for t, c in TIER_C.items():
        ax.bar(0, 0, color=c, label=f"{t} tier")
    ax.legend(fontsize=8, loc="upper right")
    ax.axhline(8, color=C["dark"], ls=":", lw=1)
    ax.text(len(fr) - 0.5, 9, "cross-validation floor (n=8)", ha="right",
            fontsize=8, color=C["dark"])

    # (b) exam types
    ax = fig.add_subplot(gs[1, 0])
    if "exam_type" in df.columns:
        vc = df["exam_type"].value_counts().head(8)[::-1]
        ax.barh(vc.index, vc.values, color=C["green"])
        ax.set_xlabel("studies")
        ax.set_title("(b) Examination type")
    else:
        ax.axis("off")

    # (c) fold balance
    ax = fig.add_subplot(gs[1, 1])
    if folds is not None and Y is not None:
        mat = np.array([[Y[folds == f, j].sum() for j in range(len(TARGETS))]
                        for f in sorted(set(folds))])
        im = ax.imshow(mat, cmap="Blues", aspect="auto")
        ax.set_yticks(range(mat.shape[0]))
        ax.set_yticklabels([f"fold {f}" for f in sorted(set(folds))])
        ax.set_xticks(range(len(TARGETS)))
        ax.set_xticklabels(TARGETS, rotation=45, ha="right", fontsize=7)
        ax.grid(False)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j, i, int(mat[i, j]), ha="center", va="center", fontsize=7,
                        color="white" if mat[i, j] > mat.max() * 0.55 else C["dark"])
        ax.set_title("(c) Positives per fold")
    save(fig, "fig01_dataset_overview",
         "Dataset composition, class imbalance and fold balance.")
else:
    skip("Dataset overview", "label_frequency.csv / labels_multihot.csv")

  saved fig01_dataset_overview


In [4]:
if "report_clean" in df.columns or "report_proc" in df.columns:
    col = "report_proc" if "report_proc" in df.columns else "report_clean"
    lens = df[col].fillna("").str.split().str.len()
    imp_len = df["impression"].fillna("").str.split().str.len() if "impression" in df else None
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].hist(lens, bins=28, color=C["primary"])
    ax[0].axvline(lens.median(), color=C["accent"], ls="--",
                  label=f"median {int(lens.median())}")
    ax[0].set_xlabel("words per report")
    ax[0].set_ylabel("studies")
    ax[0].set_title("Full report length")
    ax[0].legend(fontsize=8)
    if imp_len is not None:
        ax[1].hist(imp_len, bins=28, color=C["green"])
        ax[1].axvline(imp_len.median(), color=C["accent"], ls="--",
                      label=f"median {int(imp_len.median())}")
        ax[1].set_xlabel("words in impression")
        ax[1].set_title("Impression length — the generation target")
        ax[1].legend(fontsize=8)
    plt.tight_layout()
    save(fig, "fig02_report_lengths",
         "Report and impression length distributions.")

  saved fig02_report_lengths


## 2. Sample frames

One figure showing what the model actually sees after panel decomposition and
enhancement. Include this early in the paper — readers unfamiliar with
photographed console output will otherwise misjudge the difficulty.

In [5]:
import cv2


def resolve_frames(frames_df):
    if frames_df is None:
        return None
    s = Path(str(frames_df["frame_path"].iloc[0]))
    if s.exists():
        return frames_df
    rel = Path(s.parent.name) / s.name
    cands = [Path("/kaggle/working/frames")]
    for d in ("*/frames", "*/*/frames", "*/frames/*"):
        cands += sorted(Path("/kaggle/input").glob(d))
    root = next((c for c in cands if (c / rel).exists()), None)
    if root is None:
        zips = sorted(Path("/kaggle/input").rglob("frames.zip"))
        if zips:
            dest = Path("/kaggle/working/frames")
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(dest)
            if (dest / rel).exists():
                root = dest
    if root is None:
        return None
    frames_df = frames_df.copy()
    frames_df["frame_path"] = [str(root / Path(p).parent.name / Path(p).name)
                               for p in frames_df["frame_path"]]
    return frames_df


frames = resolve_frames(frames)
if frames is not None:
    want = ["Fatty liver", "Cholelithiasis", "Normal pregnancy", "Cystitis",
            "Bulky uterus", "Normal study"]
    have = [l for l in want if (df["primary_label"] == l).any()][:6]
    if have:
        fig, axes = plt.subplots(2, len(have), figsize=(2.1 * len(have), 4.6))
        axes = np.atleast_2d(axes)
        for c, lab in enumerate(have):
            sid = df.loc[df["primary_label"] == lab, "study_id"].iloc[0]
            fr = frames[frames["study_id"] == sid]["frame_path"].tolist()[:2]
            for r in range(2):
                ax = axes[r, c]
                ax.axis("off")
                if r < len(fr):
                    img = cv2.imread(fr[r])
                    if img is not None:
                        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                if r == 0:
                    ax.set_title("\n".join(textwrap.wrap(lab, 14)), fontsize=8.5)
        fig.suptitle("Decomposed and enhanced frames by finding", y=1.02, fontsize=11)
        plt.tight_layout()
        save(fig, "fig03_sample_frames",
             "Representative preprocessed frames for the most common findings.")
else:
    skip("Sample frames", "frame images (frames/ or frames.zip)")

  saved fig03_sample_frames


## 3. Model comparison

The main result. Confidence intervals come from `results_with_ci.csv`
(notebook 06); without it the bars are computed here without intervals, which
you should not publish.

In [6]:
ci = load_csv("results_with_ci.csv")
if ci is not None and len(ci):
    d = ci.sort_values("macro_F1").reset_index(drop=True)

    def bounds(col, point):
        """Parse '[lo, hi]' from the sorted table, not the original order."""
        lo = np.array([float(v.strip("[]").split(",")[0]) for v in d[col]])
        hi = np.array([float(v.strip("[]").split(",")[1]) for v in d[col]])
        # The bootstrap mean can sit marginally outside the percentile interval;
        # clip so the error bars stay non-negative.
        return np.minimum(lo, point), np.maximum(hi, point)

    lo_ma, hi_ma = bounds("macro_F1_95CI", d["macro_F1"].values)
    lo_mi, hi_mi = bounds("micro_F1_95CI", d["micro_F1"].values)
    ypos = np.arange(len(d))

    fig, ax = plt.subplots(1, 2, figsize=(13, 0.45 * len(d) + 2.6))
    for a, (val, lo, hi, ttl) in zip(ax, [
            (d["macro_F1"].values, lo_ma, hi_ma, "macro-F1 (rare classes count equally)"),
            (d["micro_F1"].values, lo_mi, hi_mi, "micro-F1 (common classes dominate)")]):
        base = [C["grey"] if m in ("prior", "majority") else
                (C["accent"] if m == BEST else C["primary"]) for m in d["model"]]
        a.barh(ypos, val, color=base, height=0.62)
        a.errorbar(val, ypos, xerr=[val - lo, hi - val], fmt="none",
                   ecolor=C["dark"], capsize=3, lw=1.1)
        a.set_yticks(ypos)
        a.set_yticklabels([nice(m) for m in d["model"]])
        a.set_xlabel("F1")
        a.set_title(ttl)
        a.set_xlim(0, min(1.06, max(hi) * 1.22))
        for i, (v, h) in enumerate(zip(val, hi)):
            a.text(h + 0.018, i, f"{v:.3f}", va="center", fontsize=8,
                   color=C["dark"])
    fig.suptitle("Classification performance, 5-fold out-of-fold with 95% bootstrap CI",
                 y=1.0)
    plt.tight_layout()
    save(fig, "fig04_model_comparison",
         "Main classification result with bootstrap confidence intervals. "
         "Grey bars are non-learning baselines.")
    print("\nOverlapping intervals mean the difference is not established — "
          "check mcnemar_tests.csv before claiming one model beats another.")
elif MODELS:
    from sklearn.metrics import f1_score
    rows = [{"model": m,
             "macro_F1": f1_score(Y, binarise(m), average="macro", zero_division=0),
             "micro_F1": f1_score(Y, binarise(m), average="micro", zero_division=0)}
            for m in MODELS]
    d = pd.DataFrame(rows).sort_values("macro_F1")
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(d) + 2))
    ax.barh(d["model"].map(nice), d["macro_F1"], color=C["primary"])
    ax.set_xlabel("macro-F1")
    ax.set_title("Model comparison (no CI — run notebook 06)")
    save(fig, "fig04_model_comparison", "Model comparison without intervals.")
else:
    skip("Model comparison", "oof_*.npy")

  saved fig04_model_comparison

Overlapping intervals mean the difference is not established — check mcnemar_tests.csv before claiming one model beats another.


## 4. ROC and precision–recall curves

PR curves are the more informative pair here: with prevalences from 2% to
32%, ROC-AUC flatters every model. Report both, discuss PR.

In [7]:
if BEST and Y is not None:
    from sklearn.metrics import roc_curve, precision_recall_curve, auc, average_precision_score
    prob = MODELS[BEST]
    keep = [j for j in range(len(TARGETS)) if 0 < Y[:, j].sum() < len(Y)]
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 5))
    cmap = plt.cm.viridis(np.linspace(0, 0.9, len(keep)))
    for c, j in enumerate(keep):
        fpr, tpr, _ = roc_curve(Y[:, j], prob[:, j])
        ax[0].plot(fpr, tpr, color=cmap[c], lw=1.4,
                   label=f"{TARGETS[j]} ({auc(fpr,tpr):.2f})")
        pr, rc, _ = precision_recall_curve(Y[:, j], prob[:, j])
        ax[1].plot(rc, pr, color=cmap[c], lw=1.4,
                   label=f"{TARGETS[j]} ({average_precision_score(Y[:,j],prob[:,j]):.2f})")
        ax[1].axhline(Y[:, j].sum() / len(Y), color=cmap[c], lw=0.6, alpha=0.35, ls=":")
    ax[0].plot([0, 1], [0, 1], color=C["grey"], ls="--", lw=1)
    ax[0].set_xlabel("false positive rate")
    ax[0].set_ylabel("true positive rate")
    ax[0].set_title(f"ROC — {nice(BEST)} (AUC in legend)")
    ax[1].set_xlabel("recall")
    ax[1].set_ylabel("precision")
    ax[1].set_title("Precision–recall (AP in legend; dotted = class prevalence)")
    ax[1].legend(fontsize=7, loc="upper right", ncol=1)
    plt.tight_layout()
    save(fig, "fig05_roc_pr",
         "Per-class ROC and precision-recall curves for the best model.")
else:
    skip("ROC/PR curves", "oof_*.npy")

  saved fig05_roc_pr


## 5. Where each model works

Per-class F1 across every model at once. Read the columns: a class that is
dark for all models is one no modality can see, which is a finding for the
discussion rather than a failure to hide.

In [8]:
if MODELS and Y is not None:
    from sklearn.metrics import f1_score
    order = sorted(MODELS, key=lambda m: -f1_score(Y, binarise(m), average="macro",
                                                   zero_division=0))
    mat = np.array([[f1_score(Y[:, j], binarise(m)[:, j], zero_division=0)
                     for j in range(len(TARGETS))] for m in order])
    fig, ax = plt.subplots(figsize=(1.0 * len(TARGETS) + 2, 0.5 * len(order) + 2.2))
    im = ax.imshow(mat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(TARGETS)))
    ax.set_xticklabels([f"{t}\n(n={int(Y[:,j].sum())})" for j, t in enumerate(TARGETS)],
                       rotation=45, ha="right", fontsize=7.5)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([nice(m) for m in order])
    ax.grid(False)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if (v < 0.22 or v > 0.82) else C["dark"])
    ax.set_title("Per-class F1 by model")
    fig.colorbar(im, ax=ax, shrink=0.75, label="F1")
    plt.tight_layout()
    save(fig, "fig06_per_class_f1", "Per-class F1 across all models.")

  saved fig06_per_class_f1


## 6. Error decomposition

Splitting errors into false positives and false negatives tells you what kind
of mistake the model makes. A model that is precision-heavy on rare classes
is behaving like a prior; one that is recall-heavy is over-calling.

In [9]:
if BEST and Y is not None:
    pred = binarise(BEST)
    tp = (pred * Y).sum(0)
    fp = (pred * (1 - Y)).sum(0)
    fn = ((1 - pred) * Y).sum(0)
    o = np.argsort(-(tp + fn))
    fig, ax = plt.subplots(figsize=(11, 4.4))
    x = np.arange(len(TARGETS))
    ax.bar(x, tp[o], color=C["green"], label="true positive")
    ax.bar(x, fn[o], bottom=tp[o], color=C["accent"], label="false negative (missed)")
    ax.bar(x, -fp[o], color=C["amber"], label="false positive (over-called)")
    ax.axhline(0, color=C["dark"], lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([TARGETS[j] for j in o], rotation=42, ha="right", fontsize=8)
    ax.set_ylabel("studies")
    ax.set_title(f"Error decomposition — {nice(BEST)}")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save(fig, "fig07_error_decomposition",
         "True positives, misses and over-calls per class.")

    # Which studies does every model get wrong?
    hard = np.zeros(len(df))
    for m in MODELS:
        if m in ("prior", "majority"):
            continue
        hard += (binarise(m) != Y).any(1).astype(int)
    n_real = max(1, len([m for m in MODELS if m not in ("prior", "majority")]))
    hard_df = pd.DataFrame({
        "study_id": study_ids,
        "models_wrong": hard.astype(int),
        "primary_label": df["primary_label"] if "primary_label" in df else "",
        "impression": df["impression"].fillna("").str[:80] if "impression" in df else "",
    }).sort_values("models_wrong", ascending=False)
    hard_df.to_csv(ART / "hardest_studies.csv", index=False)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    vc = hard_df["models_wrong"].value_counts().sort_index()
    ax.bar(vc.index, vc.values, color=C["primary"])
    ax.set_xlabel(f"number of models that got the study wrong (of {n_real})")
    ax.set_ylabel("studies")
    ax.set_title("Difficulty spectrum")
    plt.tight_layout()
    save(fig, "fig08_difficulty", "How many models fail on each study.")
    print("\nHardest studies (every model wrong):")
    print(hard_df[hard_df["models_wrong"] == n_real].head(8).to_string(index=False))

  saved fig07_error_decomposition
  saved fig08_difficulty

Hardest studies (every model wrong):
study_id  models_wrong      primary_label                                                                     impression
    R172            12       Ovarian cyst                                                Bilateral multi-cystic ovaries.
    R186            12       Bulky uterus                                                                  Bulky uterus.
     R58            12       Bulky uterus                                  Just bulky uterus with echogenic endometrium.
    R177            12           Cystitis                                                                      Cystitis.
    R200            12    Significant PVR 1. Mildly echogenic both kidneys. 2. Mild cystitis. 3. MCC 173 cc & PVR 21 cc.
     R79            12    Significant PVR                                   1. Bulky uterus. 2. MCC- 494 cc, PVR- 21 cc.
     R76            12  Pelvic collection   Normal size 

## 7. Score distributions

Whether the model separates positives from negatives at all, before any
threshold is applied. Heavy overlap with a good AUC means the ranking works
but the operating point is arbitrary — worth saying explicitly.

In [10]:
if BEST and Y is not None:
    keep = [j for j in range(len(TARGETS)) if Y[:, j].sum() >= 5][:6]
    if keep:
        fig, axes = plt.subplots(2, 3, figsize=(13, 6))
        for ax, j in zip(axes.ravel(), keep):
            pos = MODELS[BEST][Y[:, j] == 1, j]
            neg = MODELS[BEST][Y[:, j] == 0, j]
            bins = np.linspace(0, 1, 26)
            ax.hist(neg, bins=bins, color=C["grey"], alpha=0.75, label="negative")
            ax.hist(pos, bins=bins, color=C["accent"], alpha=0.8, label="positive")
            ax.axvline(THRESH[BEST][j], color=C["dark"], ls="--", lw=1.2,
                       label=f"threshold {THRESH[BEST][j]:.2f}")
            ax.set_title(f"{TARGETS[j]} (n={int(Y[:,j].sum())})", fontsize=9.5)
            ax.set_yscale("log")
            ax.legend(fontsize=7)
        for ax in axes.ravel()[len(keep):]:
            ax.axis("off")
        fig.suptitle(f"Predicted score distributions — {nice(BEST)}", y=1.01)
        plt.tight_layout()
        save(fig, "fig09_score_distributions",
             "Positive vs negative score separation with tuned thresholds.")

  saved fig09_score_distributions


## 8. Retrieval quality

In [11]:
ret = load_csv("retrieval_metrics.csv")
if ret is not None and len(ret):
    ks = [int(c.split("@")[1]) for c in ret.columns if c.startswith("Recall@")]
    ks = sorted(ks)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    for _, row in ret.iterrows():
        is_rand = "random" in str(row["method"]).lower()
        ax[0].plot(ks, [row[f"Recall@{k}"] for k in ks], "o-" if not is_rand else "s--",
                   color=C["grey"] if is_rand else C["primary"],
                   lw=2 if not is_rand else 1.3, label=str(row["method"])[:38])
        ax[1].plot(ks, [row[f"Precision@{k}"] for k in ks], "o-" if not is_rand else "s--",
                   color=C["grey"] if is_rand else C["green"],
                   lw=2 if not is_rand else 1.3, label=str(row["method"])[:38])
    ax[0].set_xlabel("K")
    ax[0].set_ylabel("Recall@K")
    ax[0].set_title("Does a clinically matching case appear in the top K?")
    ax[0].legend(fontsize=8)
    ax[0].set_ylim(0, 1.02)
    ax[1].set_xlabel("K")
    ax[1].set_ylabel("Precision@K")
    ax[1].set_title("What fraction of retrieved cases match?")
    ax[1].legend(fontsize=8)
    ax[1].set_ylim(0, 1.02)
    plt.tight_layout()
    save(fig, "fig10_retrieval",
         "Retrieval quality against a random-retrieval baseline.")
    if "MRR" in ret.columns:
        print(ret[["method", "MRR"]].to_string(index=False))
else:
    skip("Retrieval quality", "retrieval_metrics.csv")

  saved fig10_retrieval
                    method    MRR
BiomedCLIP image embedding 0.5200
           random baseline 0.3699


## 9. Report generation

Clinical efficacy and surface overlap plotted side by side, because they
disagree — and that disagreement is one of the more interesting things you
can report.

In [12]:
gen = load_csv("generation_metrics.csv")
gci = load_csv("generation_ci.csv")
if gen is not None and len(gen):
    g = gen.sort_values("CE_F1").reset_index(drop=True)
    ypos = np.arange(len(g))
    fig, ax = plt.subplots(1, 2, figsize=(13, 0.5 * len(g) + 2.6))

    cols = [C["grey"] if m in ("constant_normal", "random_train_report")
            else C["primary"] for m in g["method"]]
    ax[0].barh(ypos, g["CE_F1"], color=cols, height=0.6)
    if gci is not None:
        m = gci.set_index("method")
        lo = [m["CI_low"].get(k, np.nan) for k in g["method"]]
        hi = [m["CI_high"].get(k, np.nan) for k in g["method"]]
        ax[0].errorbar(g["CE_F1"], ypos,
                       xerr=[g["CE_F1"] - np.array(lo), np.array(hi) - g["CE_F1"]],
                       fmt="none", ecolor=C["dark"], capsize=3, lw=1.1)
    ax[0].set_yticks(ypos)
    ax[0].set_yticklabels(g["method"])
    ax[0].set_xlabel("clinical efficacy F1")
    ax[0].set_title("Does the text state the right finding?")
    _right = np.array(hi) if gci is not None else g["CE_F1"].values
    _right = np.where(np.isfinite(_right), _right, g["CE_F1"].values)
    ax[0].set_xlim(0, float(np.nanmax(_right)) * 1.25)
    for i, (v, h) in enumerate(zip(g["CE_F1"], _right)):
        ax[0].text(h + 0.012, i, f"{v:.3f}", va="center", fontsize=8,
                   color=C["dark"])

    surf = [c for c in ("BLEU-4", "ROUGE-L", "METEOR") if c in g.columns]
    w = 0.26
    for k, c in enumerate(surf):
        ax[1].barh(ypos + (k - 1) * w, g[c], height=w,
                   color=[C["primary"], C["green"], C["amber"]][k], label=c)
    ax[1].set_yticks(ypos)
    ax[1].set_yticklabels([])
    ax[1].set_xlabel("score")
    ax[1].set_title("Surface overlap with the reference")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    save(fig, "fig11_generation",
         "Report generation: clinical efficacy versus surface-overlap metrics.")

    if "constant_normal" in set(g["method"]):
        cn = g.loc[g["method"] == "constant_normal"].iloc[0]
        print(f"Reference point — always writing 'Normal USG findings.' scores "
              f"CE-F1 {cn['CE_F1']:.3f}"
              + (f", ROUGE-L {cn['ROUGE-L']:.3f}" if "ROUGE-L" in cn else "")
              + ".\nAny method that does not clearly beat this has not learned "
                "to report findings.")
else:
    skip("Generation metrics", "generation_metrics.csv")

  saved fig11_generation
Reference point — always writing 'Normal USG findings.' scores CE-F1 0.317, ROUGE-L 0.159.
Any method that does not clearly beat this has not learned to report findings.


In [13]:
sweep = load_csv("rag_k_sweep.csv")
if sweep is not None and len(sweep):
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(sweep["K"], sweep["CE_F1"], "o-", color=C["accent"], lw=2, label="CE F1")
    ax.plot(sweep["K"], sweep["CE_precision"], "s--", color=C["primary"], label="CE precision")
    ax.plot(sweep["K"], sweep["CE_recall"], "^--", color=C["green"], label="CE recall")
    if "Recall@K" in sweep.columns:
        ax2 = ax.twinx()
        ax2.plot(sweep["K"], sweep["Recall@K"], ":", color=C["grey"], lw=1.6,
                 label="Recall@K (retrieval)")
        ax2.set_ylabel("Recall@K", color=C["grey"])
        ax2.grid(False)
    ax.set_xlabel("K retrieved cases")
    ax.set_ylabel("clinical efficacy")
    ax.set_title("Effect of K on retrieval-augmented generation")
    ax.legend(fontsize=8, loc="lower right")
    best_k = sweep.loc[sweep["CE_F1"].idxmax(), "K"]
    ax.axvline(best_k, color=C["dark"], ls=":", lw=1)
    ax.text(best_k, ax.get_ylim()[0], f" best K={int(best_k)}", fontsize=8)
    plt.tight_layout()
    save(fig, "fig12_k_sweep", "Ablation over the number of retrieved cases.")

  saved fig12_k_sweep


## 10. Embedding space

In [14]:
emb_p = find_file("study_image_embeddings.npy")
if emb_p is not None:
    Z = np.load(emb_p)
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    try:
        from sklearn.manifold import TSNE
        from sklearn.decomposition import PCA
        e2 = TSNE(n_components=2, perplexity=min(20, len(Z) // 6), init="pca",
                  random_state=42).fit_transform(Z)
        p2 = PCA(2).fit_transform(Z)
        grp = (df["primary_label_grouped"].fillna("Other").values
               if "primary_label_grouped" in df else np.array(["all"] * len(df)))
        fig, ax = plt.subplots(1, 2, figsize=(13, 5.4))
        for a, pts, ttl in [(ax[0], e2, "t-SNE"), (ax[1], p2, "PCA")]:
            for g in pd.unique(grp):
                k = grp == g
                a.scatter(pts[k, 0], pts[k, 1], s=26, alpha=0.82, label=g)
            a.set_xticks([])
            a.set_yticks([])
            a.set_title(f"{ttl} of study image embeddings")
        ax[1].legend(fontsize=8, loc="best")
        plt.tight_layout()
        save(fig, "fig13_embedding_space",
             "BiomedCLIP study embeddings coloured by anatomical group.")
    except Exception as e:
        print("Embedding projection skipped:", e)
else:
    skip("Embedding space", "study_image_embeddings.npy (notebook 04)")

  saved fig13_embedding_space


In [15]:
sep = load_csv("embedding_separability.csv")
if sep is not None and len(sep):
    s = sep.sort_values("gap")
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(s) + 2))
    ax.barh(s["label"], s["gap"],
            color=[C["green"] if g > 0 else C["accent"] for g in s["gap"]])
    ax.axvline(0, color=C["dark"], lw=1)
    ax.set_xlabel("within-class similarity minus between-class similarity")
    ax.set_title("Does the frozen embedding space already separate each finding?")
    plt.tight_layout()
    save(fig, "fig14_separability",
         "Embedding separability per class before any fine-tuning.")

  saved fig14_separability


## 11. MIL attention

Whether the attention pooling actually selected frames or spread itself
evenly. If concentration sits near zero, do not present attention maps as
evidence in the paper.

In [16]:
att_files = list(ART.glob("attention_*.npy")) + \
            [p for p in Path("/kaggle/input").rglob("attention_*.npy")]
adf = load_csv("mil_attention_analysis.csv")
if adf is None and att_files and frames is not None:
    att = np.load(att_files[0], allow_pickle=True).item()
    rows = []
    for sid, a in att.items():
        n = len(frames[frames["study_id"] == sid])
        if n == 0:
            continue
        w = np.asarray(a).ravel()[:n]
        if w.sum() <= 0:
            continue
        w = w / w.sum()
        rows.append({"study_id": sid, "n_frames": n, "top_weight": float(w.max()),
                     "entropy": float(-(w * np.log(w + 1e-9)).sum()),
                     "uniform_entropy": float(np.log(max(n, 2)))})
    adf = pd.DataFrame(rows)
    if len(adf):
        adf["concentration"] = 1 - adf["entropy"] / adf["uniform_entropy"].clip(lower=1e-6)

if adf is not None and len(adf):
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    ax[0].hist(adf["top_weight"], bins=22, color=C["primary"])
    ax[0].set_title("Attention mass on the top frame")
    ax[0].set_xlabel("weight")
    ax[1].hist(adf["concentration"], bins=22, color=C["accent"])
    ax[1].set_title("Concentration (0 = uniform, 1 = one frame)")
    ax[1].set_xlabel("concentration")
    ax[2].scatter(adf["n_frames"], adf["top_weight"], s=22, alpha=0.6,
                  color=C["green"])
    ax[2].plot(sorted(adf["n_frames"].unique()),
               [1 / n for n in sorted(adf["n_frames"].unique())], "--",
               color=C["grey"], label="uniform baseline")
    ax[2].set_xlabel("frames in study")
    ax[2].set_ylabel("top-frame weight")
    ax[2].set_title("Selectivity vs bag size")
    ax[2].legend(fontsize=8)
    plt.tight_layout()
    save(fig, "fig15_mil_attention",
         "Attention concentration of the MIL pooling layer.")
    print(f"Mean concentration {adf['concentration'].mean():.3f} — "
          + ("attention is selective." if adf["concentration"].mean() > 0.15
             else "attention is close to uniform; treat it as decorative."))
else:
    skip("MIL attention", "attention_*.npy (notebook 03 image models)")

  saved fig15_mil_attention
Mean concentration 0.855 — attention is selective.


## 12. Qualitative examples

Rendered as an image so it drops straight into slides. Keep two successes and
two failures; a reviewer who sees only successes assumes you cherry-picked.

In [17]:
cases = load_csv("rag_case_studies.csv")
if cases is None:
    gr = load_csv("generated_reports.csv")
    if gr is not None:
        meth = [c for c in gr.columns if c not in ("study_id", "reference")]
        pick = "consensus_fusion@K" if "consensus_fusion@K" in meth else meth[0]
        gr["match"] = gr["reference"].str.lower().str[:18] == gr[pick].str.lower().str[:18]
        cases = pd.concat([gr[gr["match"]].head(2), gr[~gr["match"]].head(2)])
        cases = cases.rename(columns={pick: "generated"})
        cases["outcome"] = np.where(cases["match"], "correct", "incorrect")

if cases is not None and len(cases):
    rows = cases.head(4)
    fig, axes = plt.subplots(len(rows), 1, figsize=(11.5, 1.75 * len(rows)))
    axes = np.atleast_1d(axes)
    for ax, (_, r) in zip(axes, rows.iterrows()):
        ax.axis("off")
        ok = str(r.get("outcome", "")) == "correct"
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                   facecolor="#F2F7F3" if ok else "#FBF1EF",
                                   edgecolor=C["green"] if ok else C["accent"], lw=1.4))
        head = f"{r.get('study_id','')}   [{ 'correct' if ok else 'incorrect' }]"
        ax.text(0.015, 0.80, head, fontsize=9.5, weight="bold", transform=ax.transAxes,
                color=C["green"] if ok else C["accent"])
        ref = textwrap.shorten(str(r.get("reference", "")), 150, placeholder=" …")
        gen = textwrap.shorten(str(r.get("generated", "")), 150, placeholder=" …")
        ax.text(0.015, 0.50, f"reference:  {ref}", fontsize=8.6, transform=ax.transAxes)
        ax.text(0.015, 0.20, f"generated:  {gen}", fontsize=8.6, transform=ax.transAxes)
    fig.suptitle("Retrieval-augmented generation — successes and failures", y=1.01)
    plt.tight_layout()
    save(fig, "fig16_qualitative", "Qualitative report-generation examples.")
else:
    skip("Qualitative examples", "rag_case_studies.csv / generated_reports.csv")

  saved fig16_qualitative


In [18]:
attr = load_csv("text_attribution.csv")
if attr is not None and len(attr):
    sub = attr.head(6)
    fig, axes = plt.subplots(2, 3, figsize=(14, 5.6))
    for ax, (_, r) in zip(axes.ravel(), sub.iterrows()):
        terms = str(r["top_terms"]).split("|")[:7][::-1]
        ax.barh(range(len(terms)), np.linspace(0.4, 1.0, len(terms)),
                color=C["primary"])
        ax.set_yticks(range(len(terms)))
        ax.set_yticklabels(terms, fontsize=8)
        ax.set_xticks([])
        ax.set_title(r["label"], fontsize=9.5)
    for ax in axes.ravel()[len(sub):]:
        ax.axis("off")
    fig.suptitle("Terms driving each class in the linear text model "
                 "(rank order, not magnitude)", y=1.01)
    plt.tight_layout()
    save(fig, "fig17_text_attribution",
         "Highest-weighted findings terms per class.")
    print("Check these for shortcut learning: terms describing the referral "
          "reason rather than\nthe sonographic appearance will not generalise "
          "to another clinic.")

  saved fig17_text_attribution
Check these for shortcut learning: terms describing the referral reason rather than
the sonographic appearance will not generalise to another clinic.


## Figure index and download

In [19]:
manifest = pd.DataFrame(SAVED)
manifest.to_csv(ART / "figure_manifest.csv", index=False)
print(manifest.to_string(index=False))

zip_path = Path("/kaggle/working/all_figures.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIG.glob("*.png")) + sorted(FIG.glob("*.pdf")):
        zf.write(p, arcname=p.name)
    for name in ("results_with_ci.csv", "generation_metrics.csv",
                 "retrieval_metrics.csv", "rag_k_sweep.csv",
                 "mcnemar_tests.csv", "ablation_classification.csv",
                 "baseline_per_class.csv", "figure_manifest.csv",
                 "hardest_studies.csv"):
        p = find_file(name)
        if p is not None:
            zf.write(p, arcname=f"tables/{name}")
print(f"\n{zip_path} — {zip_path.stat().st_size/1e6:.1f} MB, "
      f"{len(SAVED)} figures plus the result tables.")
print("Download it from the Output panel on the right.")

                   figure                          file                                                                                               caption
   fig01_dataset_overview    fig01_dataset_overview.png                                                Dataset composition, class imbalance and fold balance.
     fig02_report_lengths      fig02_report_lengths.png                                                           Report and impression length distributions.
      fig03_sample_frames       fig03_sample_frames.png                                      Representative preprocessed frames for the most common findings.
   fig04_model_comparison    fig04_model_comparison.png Main classification result with bootstrap confidence intervals. Grey bars are non-learning baselines.
             fig05_roc_pr              fig05_roc_pr.png                                         Per-class ROC and precision-recall curves for the best model.
       fig06_per_class_f1        fig06_per_class_f1.

## Reading your own results

Before you write a single sentence of the results section, check these four
things in the figures above:

1. **Is the best model clearly above the grey baselines in fig04?** If the
   intervals overlap the prior baseline, you do not yet have a result — say
   so, and treat the pipeline as the contribution.
2. **In fig06, does performance track the class counts in the column
   headers?** If it does almost perfectly, the model learned prevalence
   rather than pathology. That belongs in the limitations.
3. **In fig11, does anything beat `constant_normal` on clinical efficacy by
   more than its confidence interval?** That comparison is the honest test of
   whether report generation works at all here.
4. **In fig17, are the driving terms sonographic descriptions or referral
   reasons?** The latter is shortcut learning and needs to be disclosed.

Answering all four honestly will make the paper considerably harder to
reject than any amount of additional tuning.